In [20]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [21]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def ranktoset (A):
    A = list(A)
    sets = [[[A[0]]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append([new])
    return(sets)

def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-1:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominalpowerU (sets,p,R,r,m,r_f,c,rav):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    f_obj = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=0)
    constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1)- (1-cp.sum(a))*r_f + z4 + z2 <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f <= c)  


def robustcheckpowerU(a,R,r,c,p,m,r_f,rav):    ### actually not being used
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value <= c)

In [22]:
def cutting_plane(R,r,c,p,m,r_f,sets,rav):
    nonstop = True
    iterations = 1
    while nonstop == True:
        realsets = convertlist(sets)
        #print(solvenominalpowerU(realsets,p,R,r,m,r_f,c,rav))
        [a,obj] = solvenominalpowerU(realsets,p,R,r,m,r_f,c,rav)
        newrank = np.argsort(R.dot(a))
        [sets,added] = makeset(sets, newrank)
        if robustcheck(a,R,r,c,p,m,r_f) == True:
            return(a,obj,iterations)
        iterations = iterations + 1
    

In [44]:
np.random.seed(5)

In [63]:
N=5
p = (np.zeros(N)+1)*1/N
I = 5
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.02289969 0.00642032 0.00510418 0.08897928 0.03334271]
[[-0.1294188   0.17654237  0.15703103 -0.04483048  0.18699718]
 [-0.02223884 -0.06548599 -0.1969459   0.12709598 -0.035838  ]
 [-0.06178525 -0.1798     -0.22303116 -0.1078478   0.19599196]
 [-0.11277637  0.3389719   0.13165189  0.08161303 -0.19064813]
 [ 0.44071774 -0.2381267   0.15681502  0.38886566  0.01021056]]


In [64]:
gam = 2.5
rav = 1-gam
r = 0.1
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 0.1
sets = ranktoset(np.arange(N))
print(cutting_plane(R,r,c,p,m,r_f,sets,rav))

(array([-8.75993714e-09,  2.06403264e-01, -9.10698073e-09,  3.49382930e-01,
        4.44213803e-01]), -262.0472600644408, 1)


In [65]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
print(solvenominalpowerU (psets,p,R,r,m,r_f,c,rav))
#print(robustcheckpowerU(a,R,r,c,p,m,r_f,rav))


(array([-3.51428823e-09,  2.06401218e-01, -3.57901075e-09,  3.49386640e-01,
        4.44212145e-01]), -262.04728793603476)
